In [192]:
import pandas as pd
from tqdm import tqdm
from glob import glob
from sklearn.neighbors import NearestNeighbors
import numpy as np
import meteostat as ms
from sqlalchemy import create_engine
from datetime import date
import pickle


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tsiaras/uk-road-safety-accidents-and-vehicles")

print("Path to dataset files:", path)

Download already complete (149163721 bytes).
Extracting files...
Path to dataset files: C:\Users\M\.cache\kagglehub\datasets\tsiaras\uk-road-safety-accidents-and-vehicles\versions\3


In [28]:
acc = pd.read_csv("../data/raw/Accident_Information.csv")
acc['Date'] = pd.to_datetime(acc['Date'], format='%Y-%m-%d')

C:\Users\M\AppData\Local\Temp\ipykernel_4064\1717480648.py:1: DtypeWarning: Columns (0: Accident_Index) have mixed types. Specify dtype option on import or set low_memory=False.
  acc = pd.read_csv("../data/raw/Accident_Information.csv")


In [201]:
veh = pd.read_csv("../data/raw/Vehicle_Information.csv", encoding="cp1252")

steps:
1. load the stations weather data 
2. load the stations meta data
3. for each row get 2 things: date as year and month, longitude and latitidue and 
4. run the algorithm that will get the nearest station to this accident
5. use date to get weather data

In [21]:
engine = create_engine(r'sqlite:///C:/Users/M/Downloads/stations.db')

In [45]:
query = """
    SELECT s.id, s.longitude, s.latitude
    FROM stations s
    WHERE s.country == "GB"
"""

In [46]:
stations_df = pd.read_sql(query, engine)

In [47]:
stations_df

,id,longitude,latitude
0,EGPW0,-0.8167,60.7333
1,03649,-1.5833,51.7500
2,03160,-3.3500,55.9500
3,03768,-0.7667,51.2833
4,03495,1.3500,52.4333
...,...,...,...
171,03781,-0.0833,51.3000
172,03894,-2.6000,49.4333
173,03766,-0.7667,51.2833
174,03313,-3.5000,53.2500


In [185]:
START = date(2005, 1, 1)
END   = date(2017, 12, 31)

station_weather = {}  # { station_id: DataFrame }

failed = []

for _, row in tqdm(stations_df.iterrows(), total=len(stations_df)):
    station_id = row["id"]
    try:
        data = ms.daily(station_id, START, END).fetch().reset_index()
        data['time'] = pd.to_datetime(data['time'], format='%Y-%m-%d')
        if not data.empty:
            station_weather[station_id] = data
    except Exception as e:
        failed.append(station_id)
        continue

print(f"Fetched: {len(station_weather)} stations")
print(f"Failed:  {len(failed)} stations")

100%|██████████| 176/176 [13:35<00:00,  4.63s/it]

Fetched: 102 stations
Failed:  74 stations


In [190]:
station_weather

{'EGPW0':            time  temp  tmin  tmax  rhum  prcp  snwd  wspd  wpgt    pres  tsun  \
 0    2005-01-01   5.4   3.1   6.8    89  <NA>  <NA>  29.5  <NA>   992.2  <NA>   
 1    2005-01-02   2.4   0.0   3.8    82  <NA>  <NA>  45.4  <NA>   992.8  <NA>   
 2    2005-01-03   5.8   1.9  10.0  <NA>  <NA>  <NA>  30.7  <NA>  1004.4  <NA>   
 3    2005-01-04   6.5  <NA>  <NA>  <NA>  <NA>  <NA>  43.7  <NA>   993.9  <NA>   
 4    2005-01-19   3.5   2.3   5.8    83  <NA>  <NA>  38.2  <NA>   985.6  <NA>   
 ...         ...   ...   ...   ...   ...   ...   ...   ...   ...     ...   ...   
 4566 2017-12-10   0.7  -2.7   3.3    91  <NA>  <NA>  11.1  <NA>   979.7  <NA>   
 4567 2017-12-11   1.6  -0.7   3.0    89  <NA>  <NA>  10.4  <NA>   989.1  <NA>   
 4568 2017-12-12   1.4  -1.1   4.5    89  <NA>  <NA>  12.5  <NA>   994.9  <NA>   
 4569 2017-12-13   3.2   1.2   5.5    91  <NA>  <NA>  20.1  <NA>   977.4  <NA>   
 4570 2017-12-14   4.2   2.3   5.9    87  <NA>  <NA>  23.6  <NA>   979.4  <NA>   
 
     

In [ ]:
# Save
with open("../data/raw/station_weather.pkl", "wb") as f:
    pickle.dump(station_weather, f)


In [193]:
with open("../data/raw/station_weather.pkl", "rb") as f:
    station_weather = pickle.load(f)

In [194]:
from sklearn.neighbors import BallTree
import numpy as np

coords_rad = np.radians(stations_df[["latitude", "longitude"]].values)
tree = BallTree(coords_rad, metric="haversine")


In [196]:
# Lookup function
def get_weather_for_accident(lat, lon, date):

    if pd.isna(lat) or pd.isna(lon):
        return {}
    
    # Find nearest station
    point = np.radians([[lat, lon]])
    dist, idx = tree.query(point, k=1)
    station_id = stations_df.iloc[idx[0][0]]["id"]
    
    # Lookup weather
    weather = station_weather.get(station_id)
    
    if weather is None or weather.empty:
        return {}
    
    # Match by date
    match =  weather[weather.time == date]
    #print(match)
    
    return match.iloc[0].to_dict() if not match.empty else {}

In [197]:
weather = get_weather_for_accident(acc.iloc[0].Latitude, acc.iloc[0].Longitude, acc.iloc[0].Date)
weather

{'time': Timestamp('2005-01-04 00:00:00'),
 'temp': 10.2,
 'tmin': 7.0,
 'tmax': 11.7,
 'rhum': 70,
 'prcp': None,
 'snwd': None,
 'wspd': 23.0,
 'wpgt': None,
 'pres': 1023.1,
 'tsun': None,
 'cldc': 6}

In [198]:
# --- Main loop ---
results = []
for i, row in tqdm(acc.iterrows(), total=len(acc)):
    record = {"Accident_Index": str(row["Accident_Index"])}
    weather = get_weather_for_accident(row["Latitude"], row["Longitude"], row["Date"])
    record.update(weather)
    results.append(record)

weather_df = pd.DataFrame(results)
#acc_merged = acc.merge(weather_df, on="Accident_Index", how="left")

100%|██████████| 2047256/2047256 [37:15<00:00, 915.61it/s]  


In [200]:
weather_df.isna().sum()

Accident_Index          0
time               994718
temp               995452
tmin              1164604
tmax              1163023
rhum              1008671
prcp              1938177
snwd              2020733
wspd              1059552
wpgt              2047256
pres              1506864
tsun              2047256
cldc              1733253
dtype: int64

In [202]:
acc_merged = acc.merge(veh, on="Accident_Index", how="left").merge(weather_df, on="Accident_Index", how="left")

In [205]:
with open("../data/interim/merged_data.pkl", "wb") as f:
    pickle.dump(acc_merged, f)

In [ ]:
#draft
# X = np.array(stations_df[["longitude", "latitude"]])
#nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(X)
#valid_mask = acc[["Longitude", "Latitude"]].notna().all(axis=1)
#valid_coords = acc.loc[valid_mask, ["Longitude", "Latitude"]]

#distances, indices = nbrs.kneighbors(np.array(valid_coords[["Longitude", "Latitude"]]))